In [25]:
%pip install pypdf
%pip install -U sentence-transformers


from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

import glob
import os




Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [26]:
### READING AVAILABLE RESUME

resume_texts = []
resume_names = []

for file in glob.glob("../resources/resumes/*.pdf"):

    text = ""

    reader = PdfReader(file)

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + " "

    resume_texts.append(text)
    resume_names.append(os.path.basename(file))

print(len(resume_texts))

5000


In [27]:
### READING AVAILABLE JDs

reader = PdfReader("../resources/JD/20_Job_Descriptions.pdf")

jd_data = []

for page_num, page in enumerate(reader.pages, start=1):

    jd_text = page.extract_text()

    jd_data.append({
        "jd_pdf": "JD_Master.pdf",
        "jd_page": page_num,
        "jd_id": f"JD_{page_num}",
        "text": jd_text
    })

jd_texts = [jd["text"] for jd in jd_data]

In [28]:
### CREATING MODEL

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [29]:
### JD EMBEDDINGS

jd_embeddings = model.encode(
    jd_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [30]:
### RESUME EMBEDDINGS

resume_embeddings = model.encode(
    resume_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

In [37]:
### CREATING SIMILARITY MATRIX

from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(
    jd_embeddings,
    resume_embeddings
)


print(jd_embeddings.shape)
print(resume_embeddings.shape)
print(similarity_matrix.shape)

(20, 384)
(5000, 384)
(20, 5000)


In [38]:
import numpy as np
import pandas as pd

rows = []

for jd_idx, jd in enumerate(jd_data):

    scores = similarity_matrix[jd_idx]

    top5_idx = np.argsort(scores)[::-1][:5]   ###argsort returns in ascending, hence [::-1]

    rows.append({
        "JD_ID": jd["jd_id"],
        "JD_Page": jd["jd_page"],
        "Best_Match_1": resume_names[top5_idx[0]],
        "Best_Match_2": resume_names[top5_idx[1]],
        "Best_Match_3": resume_names[top5_idx[2]],
        "Best_Match_4": resume_names[top5_idx[3]],
        "Best_Match_5": resume_names[top5_idx[4]]
    })

df = pd.DataFrame(rows)
df

,JD_ID,JD_Page,Best_Match_1,Best_Match_2,Best_Match_3,Best_Match_4,Best_Match_5
0,JD_1,1,Resume_033394_Steven_Watkins.pdf,Resume_034496_Susan_Watson.pdf,Resume_033028_Kaylee_Anderson.pdf,Resume_033674_Edgar_Watts.pdf,Resume_030521_Suzanne_Bowman.pdf
1,JD_2,2,Resume_032481_Christopher_Smith.pdf,Resume_034361_Brandon_Carter.pdf,Resume_032436_Derrick_Diaz.pdf,Resume_032194_Cindy_Ramirez.pdf,Resume_034359_John_Macias.pdf
2,JD_3,3,Resume_034496_Susan_Watson.pdf,Resume_033674_Edgar_Watts.pdf,Resume_031631_Todd_Burgess.pdf,Resume_031929_Valerie_Martin.pdf,Resume_034893_Pamela_Swanson.pdf
3,JD_4,4,Resume_034496_Susan_Watson.pdf,Resume_034175_Erin_Harmon.pdf,Resume_034893_Pamela_Swanson.pdf,Resume_033674_Edgar_Watts.pdf,Resume_031929_Valerie_Martin.pdf
4,JD_5,5,Resume_030056_Christopher_Lin.pdf,Resume_034068_Jared_Cooley.pdf,Resume_033654_James_Fernandez.pdf,Resume_034378_Jason_Harrison.pdf,Resume_030599_Michelle_Long.pdf
5,JD_6,6,Resume_033674_Edgar_Watts.pdf,Resume_033778_Brian_Diaz.pdf,Resume_031356_Teresa_Deleon.pdf,Resume_033654_James_Fernandez.pdf,Resume_031758_Brittany_Moore.pdf
6,JD_7,7,Resume_031609_Leah_Garcia.pdf,Resume_033674_Edgar_Watts.pdf,Resume_030089_Christopher_Lopez.pdf,Resume_034843_Kelli_Perez.pdf,Resume_034654_Deanna_Armstrong.pdf
7,JD_8,8,Resume_032436_Derrick_Diaz.pdf,Resume_034038_Michael_Lee.pdf,Resume_031609_Leah_Garcia.pdf,Resume_032076_Lisa_Nguyen.pdf,Resume_032194_Cindy_Ramirez.pdf
8,JD_9,9,Resume_033826_Heather_Becker.pdf,Resume_032481_Christopher_Smith.pdf,Resume_031171_Christopher_Reese.pdf,Resume_032575_Kevin_Armstrong.pdf,Resume_030056_Christopher_Lin.pdf
9,JD_10,10,Resume_032436_Derrick_Diaz.pdf,Resume_031917_Aaron_Wilson.pdf,Resume_033387_Joshua_Mitchell.pdf,Resume_034068_Jared_Cooley.pdf,Resume_031815_Jeffery_Gross.pdf
